# 📐 Redimensionamento de Fotos de Obra com Letterbox

**Profª M.Sc. Adriana Rolim** — IPOS Pós-Graduação
*Disciplina: Visão Computacional Aplicada à Construção Civil*

---

## 🎯 Por que padronizar tamanhos?

Modelos de visão computacional (U-Net, YOLO, ResNet) **exigem** que todas as imagens de entrada tenham as **mesmas dimensões**. Mas fotos de obra costumam vir misturadas:

- 📱 Smartphones diferentes → 1920×1080, 4032×3024, 3000×4000...
- 🚁 Drones → 5472×3648 ou 4000×3000
- 📷 Câmeras de canteiro → 1280×720 ou 1920×1080

Misturar tudo direto na rede neural **degrada o resultado**. Precisamos padronizar antes.

## 🤔 Resize simples × Letterbox

| Método | O que faz | Problema |
|---|---|---|
| **`cv2.resize` direto** | Estica/comprime até o tamanho alvo | **Distorce as proporções** — objetos circulares viram elípticos |
| **Letterbox** (este script) | Redimensiona mantendo proporção, **preenche bordas pretas** onde sobra | Adiciona pixels pretos, mas preserva geometria |

Para **construção civil**, letterbox é a escolha certa: precisamos que uma viga continue parecendo uma viga, sem esticamento que destrua medições visuais.

```
Original (16:9)         Resize direto (1:1)        Letterbox (1:1)
┌─────────────┐         ┌──────────┐                ┌──────────┐
│             │   →     │          │                │ ████████ │
│   PRÉDIO    │         │  PRÉDIO  │                │  PRÉDIO  │
│             │         │ (gordo)  │                │  (igual) │
└─────────────┘         └──────────┘                │ ████████ │
                                                     └──────────┘
                        ❌ distorcido                ✅ proporção mantida
```

---

## 📁 Pasta de entrada

🔗 https://drive.google.com/drive/folders/1oGblbMJE9rCZsk8jUOy56XrApuv6BNpf

## 0️⃣ Setup — Montar Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Configurações principais
PASTA_ENTRADA = '/content/drive/MyDrive/fotos_obra'
PASTA_SAIDA   = '/content/drive/MyDrive/fotos_obra_redimensionadas'
TAMANHO_ALVO  = (1920, 1080)  # (largura, altura)

import os
if os.path.exists(PASTA_ENTRADA):
    n = len([f for f in os.listdir(PASTA_ENTRADA) if f.lower().endswith(('.jpg','.jpeg','.png'))])
    print(f"✅ Pasta de entrada: {PASTA_ENTRADA}")
    print(f"   {n} imagens detectadas")
    print(f"\n📤 Pasta de saída: {PASTA_SAIDA}")
    print(f"📐 Tamanho alvo:   {TAMANHO_ALVO[0]}×{TAMANHO_ALVO[1]} (Full HD)")
else:
    print(f"⚠️ Pasta não encontrada: {PASTA_ENTRADA}")

---
## 1️⃣ Importações

In [ ]:
import os
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import base64
from datetime import datetime
from pathlib import Path

plt.rcParams['figure.dpi'] = 90
print("✅ Bibliotecas carregadas")

---
## 2️⃣ Funções do Script Original

São três funções: criar a pasta de saída, redimensionar **uma** imagem, e processar **todas**.

In [ ]:
def garantir_pasta(pasta):
    """Cria pasta se não existir."""
    os.makedirs(pasta, exist_ok=True)


def redimensionar_imagem(img, tamanho_alvo):
    """Redimensiona mantendo a proporção e aplicando letterbox se necessário."""
    largura_alvo, altura_alvo = tamanho_alvo
    h, w = img.shape[:2]

    # Calcula proporção — escolhe o menor fator para garantir que cabe
    proporcao = min(largura_alvo / w, altura_alvo / h)
    nova_largura = int(w * proporcao)
    nova_altura = int(h * proporcao)

    # Redimensiona mantendo proporção
    img_redimensionada = cv2.resize(img, (nova_largura, nova_altura),
                                     interpolation=cv2.INTER_AREA)

    # Cria imagem final com fundo preto (letterbox)
    imagem_final = cv2.copyMakeBorder(
        img_redimensionada,
        top=(altura_alvo - nova_altura) // 2,
        bottom=(altura_alvo - nova_altura + 1) // 2,
        left=(largura_alvo - nova_largura) // 2,
        right=(largura_alvo - nova_largura + 1) // 2,
        borderType=cv2.BORDER_CONSTANT,
        value=(0, 0, 0)
    )
    return imagem_final


print("✅ Funções definidas")

### 🔍 Entendendo a matemática

**Exemplo:** foto original `4000×3000` (4:3) precisa virar `1920×1080` (16:9)

```
Fator largura = 1920 / 4000 = 0.480
Fator altura  = 1080 / 3000 = 0.360
                              ↑
                  Escolhemos o MENOR (0.360)
                  para garantir que cabe inteiro

Nova dimensão = (4000×0.360, 3000×0.360) = (1440, 1080)
                                            ↑      ↑
                                       cabe em 1920   bate exato

Letterbox:
   - Largura sobrou: 1920 - 1440 = 480 pixels → 240 px de cada lado
   - Altura: 0 pixels (bateu exato)
```

Por que `cv2.INTER_AREA`? É o melhor algoritmo para **reduzir** imagens (preserva detalhes via média ponderada). Para **ampliar**, o ideal seria `INTER_CUBIC` ou `INTER_LANCZOS4`.

> ⚠️ **Cuidado:** se a foto original for **menor** que o alvo, o script vai **ampliar** ela com `INTER_AREA`, que não é ideal para upscaling. Em produção, considere usar interpolação diferente nesses casos (ou simplesmente não fazer upscaling, mantendo a foto pequena com bordas maiores).

---
## 3️⃣ Demonstração em UMA Imagem (antes do lote)

Antes de rodar em todas as fotos, vamos pegar **uma** e ver o efeito visual da transformação.

In [ ]:
# Pega a primeira imagem da pasta para demonstração
arquivos = sorted([f for f in os.listdir(PASTA_ENTRADA)
                   if f.lower().endswith(('.jpg','.jpeg','.png'))])

if not arquivos:
    print("⚠️ Nenhuma imagem encontrada na pasta de entrada")
else:
    nome_demo = arquivos[0]
    caminho_demo = os.path.join(PASTA_ENTRADA, nome_demo)
    img_orig = cv2.imread(caminho_demo)
    img_resize = redimensionar_imagem(img_orig, TAMANHO_ALVO)

    h_orig, w_orig = img_orig.shape[:2]
    h_new, w_new = img_resize.shape[:2]

    # Para comparação: o que aconteceria com resize direto (errado)
    img_resize_errado = cv2.resize(img_orig, TAMANHO_ALVO, interpolation=cv2.INTER_AREA)

    # Visualização comparativa
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    axes[0].imshow(cv2.cvtColor(img_orig, cv2.COLOR_BGR2RGB))
    axes[0].set_title(f'📸 Original\n{w_orig}×{h_orig} (proporção {w_orig/h_orig:.2f})')
    axes[0].axis('off')

    axes[1].imshow(cv2.cvtColor(img_resize_errado, cv2.COLOR_BGR2RGB))
    axes[1].set_title(f'❌ Resize direto (DISTORCE)\n{TAMANHO_ALVO[0]}×{TAMANHO_ALVO[1]}')
    axes[1].axis('off')

    axes[2].imshow(cv2.cvtColor(img_resize, cv2.COLOR_BGR2RGB))
    axes[2].set_title(f'✅ Letterbox (PRESERVA proporção)\n{w_new}×{h_new}')
    axes[2].axis('off')
    plt.tight_layout()
    plt.show()

    # Diagnóstico textual
    prop_orig = w_orig / h_orig
    prop_alvo = TAMANHO_ALVO[0] / TAMANHO_ALVO[1]
    print(f"\n🔍 Diagnóstico:")
    print(f"   Proporção original: {prop_orig:.3f}")
    print(f"   Proporção alvo:     {prop_alvo:.3f}")
    if abs(prop_orig - prop_alvo) < 0.01:
        print(f"   ✓ Proporções coincidem — sem barras pretas")
    elif prop_orig > prop_alvo:
        barras = (TAMANHO_ALVO[1] - int(h_orig * (TAMANHO_ALVO[0]/w_orig))) // 2
        print(f"   ⓘ Imagem mais larga que o alvo → barras pretas em CIMA e EMBAIXO (~{barras}px cada)")
    else:
        barras = (TAMANHO_ALVO[0] - int(w_orig * (TAMANHO_ALVO[1]/h_orig))) // 2
        print(f"   ⓘ Imagem mais alta que o alvo → barras pretas à ESQUERDA e DIREITA (~{barras}px cada)")

---
## 4️⃣ Processamento em Lote (script original)

Agora aplicamos em todas as imagens. A função abaixo é exatamente a do script original, com **uma pequena melhoria**: coleta estatísticas durante o processamento para alimentar o relatório HTML depois.

In [ ]:
def processar_pasta(pasta_entrada, pasta_saida, tamanho_alvo, coletar_estatisticas=True):
    """
    Percorre todas as imagens da pasta e aplica redimensionamento.

    Se coletar_estatisticas=True, retorna um DataFrame com info de cada imagem.
    """
    garantir_pasta(pasta_saida)
    registros = []

    arquivos = sorted([f for f in os.listdir(pasta_entrada)
                       if f.lower().endswith(('.jpg', '.jpeg', '.png'))])

    print(f"🚀 Processando {len(arquivos)} imagens...\n")

    for arquivo in arquivos:
        caminho_entrada = os.path.join(pasta_entrada, arquivo)
        img = cv2.imread(caminho_entrada)

        if img is None:
            print(f"[AVISO] Não foi possível abrir {arquivo}")
            registros.append({
                'arquivo': arquivo, 'status': 'erro',
                'h_orig': None, 'w_orig': None, 'h_new': None, 'w_new': None,
                'tam_orig_kb': None, 'tam_new_kb': None,
                'precisou_upscale': None, 'barras_px': None,
            })
            continue

        h_orig, w_orig = img.shape[:2]
        tam_orig_kb = os.path.getsize(caminho_entrada) / 1024

        img_redim = redimensionar_imagem(img, tamanho_alvo)
        caminho_saida = os.path.join(pasta_saida, arquivo)
        cv2.imwrite(caminho_saida, img_redim)
        tam_new_kb = os.path.getsize(caminho_saida) / 1024
        h_new, w_new = img_redim.shape[:2]

        # Cálculo do que aconteceu
        prop_orig = w_orig / h_orig
        prop_alvo = tamanho_alvo[0] / tamanho_alvo[1]
        precisou_upscale = (w_orig < tamanho_alvo[0]) and (h_orig < tamanho_alvo[1])
        # Barras de letterbox
        if prop_orig > prop_alvo:
            barras_px = (tamanho_alvo[1] - int(h_orig * (tamanho_alvo[0]/w_orig))) // 2
            tipo_barras = 'horizontais (topo/base)'
        elif prop_orig < prop_alvo:
            barras_px = (tamanho_alvo[0] - int(w_orig * (tamanho_alvo[1]/h_orig))) // 2
            tipo_barras = 'verticais (lados)'
        else:
            barras_px = 0
            tipo_barras = 'nenhuma'

        registros.append({
            'arquivo': arquivo, 'status': 'ok',
            'h_orig': h_orig, 'w_orig': w_orig,
            'h_new': h_new, 'w_new': w_new,
            'tam_orig_kb': round(tam_orig_kb, 1),
            'tam_new_kb': round(tam_new_kb, 1),
            'precisou_upscale': precisou_upscale,
            'barras_px': barras_px,
            'tipo_barras': tipo_barras,
        })
        print(f"[OK] {arquivo}: {w_orig}×{h_orig} → {w_new}×{h_new}  "
              f"({tam_orig_kb:.0f}KB → {tam_new_kb:.0f}KB)")

    return pd.DataFrame(registros) if coletar_estatisticas else None


# Executa o processamento
df_resultado = processar_pasta(PASTA_ENTRADA, PASTA_SAIDA, TAMANHO_ALVO)
print(f"\n✅ Processamento concluído: {(df_resultado['status']=='ok').sum()} sucessos, "
      f"{(df_resultado['status']=='erro').sum()} erros")

---
## 5️⃣ Estatísticas do Processamento

Tabela e gráficos resumindo o que aconteceu com cada imagem.

In [ ]:
# Tabela resumida
df_ok = df_resultado[df_resultado['status'] == 'ok'].copy()

print("📋 Resumo das transformações:\n")
print(df_ok[['arquivo', 'w_orig', 'h_orig', 'w_new', 'h_new',
             'tam_orig_kb', 'tam_new_kb', 'tipo_barras']].to_string(index=False))

# Estatísticas gerais
print("\n" + "="*60)
print("📊 ESTATÍSTICAS AGREGADAS")
print("="*60)
print(f"  Total processado:        {len(df_ok)} imagens")
print(f"  Tamanho total antes:     {df_ok['tam_orig_kb'].sum()/1024:.1f} MB")
print(f"  Tamanho total depois:    {df_ok['tam_new_kb'].sum()/1024:.1f} MB")

reducao = (1 - df_ok['tam_new_kb'].sum() / df_ok['tam_orig_kb'].sum()) * 100
icone = '📉' if reducao > 0 else '📈'
print(f"  {icone} Variação:             {abs(reducao):.1f}% {'redução' if reducao > 0 else 'aumento'}")

n_upscale = df_ok['precisou_upscale'].sum()
if n_upscale > 0:
    print(f"\n  ⚠️ {n_upscale} imagens precisaram de UPSCALING (foto menor que alvo)")
    print(f"     Considere revisar essas — qualidade pode estar degradada")

n_sem_barras = (df_ok['tipo_barras'] == 'nenhuma').sum()
print(f"\n  ✓ {n_sem_barras} imagens já tinham proporção {TAMANHO_ALVO[0]}:{TAMANHO_ALVO[1]} (sem barras pretas)")
print(f"  ⓘ {len(df_ok)-n_sem_barras} imagens receberam barras de letterbox")

In [ ]:
# Visualizações
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# 1. Tamanho de arquivo antes/depois
x = np.arange(len(df_ok))
axes[0].bar(x - 0.2, df_ok['tam_orig_kb'], 0.4, label='Antes', color='#c0392b')
axes[0].bar(x + 0.2, df_ok['tam_new_kb'], 0.4, label='Depois', color='#27ae60')
axes[0].set_xticks(x)
axes[0].set_xticklabels([n[:12] for n in df_ok['arquivo']], rotation=45, ha='right', fontsize=8)
axes[0].set_ylabel('Tamanho (KB)')
axes[0].set_title('Tamanho do arquivo: antes × depois')
axes[0].legend()
axes[0].grid(axis='y', alpha=0.3)

# 2. Tipo de barras de letterbox
tipos = df_ok['tipo_barras'].value_counts()
cores_pie = ['#3498db', '#e67e22', '#27ae60'][:len(tipos)]
axes[1].pie(tipos.values, labels=tipos.index, autopct='%1.0f%%',
            colors=cores_pie, startangle=90)
axes[1].set_title('Distribuição de tipos de letterbox')

# 3. Proporções originais
prop_orig = df_ok['w_orig'] / df_ok['h_orig']
prop_alvo = TAMANHO_ALVO[0] / TAMANHO_ALVO[1]
axes[2].hist(prop_orig, bins=10, color='#9b59b6', alpha=0.7, edgecolor='black')
axes[2].axvline(prop_alvo, color='red', linestyle='--', linewidth=2,
                label=f'Alvo ({prop_alvo:.2f})')
axes[2].set_xlabel('Proporção L/A')
axes[2].set_ylabel('Nº de imagens')
axes[2].set_title('Proporções originais × proporção alvo')
axes[2].legend()
axes[2].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

### 👀 Amostragem visual: 3 fotos antes × depois

Para validar que tudo correu bem, mostramos 3 amostras lado a lado.

In [ ]:
# Pega 3 amostras aleatórias
if len(df_ok) > 0:
    np.random.seed(42)
    n_amostras = min(3, len(df_ok))
    idx_amostras = np.random.choice(len(df_ok), n_amostras, replace=False)

    fig, axes = plt.subplots(n_amostras, 2, figsize=(14, 4*n_amostras))
    if n_amostras == 1:
        axes = axes.reshape(1, -1)

    for i, idx in enumerate(idx_amostras):
        nome = df_ok.iloc[idx]['arquivo']
        img_a = cv2.imread(os.path.join(PASTA_ENTRADA, nome))
        img_d = cv2.imread(os.path.join(PASTA_SAIDA, nome))

        axes[i, 0].imshow(cv2.cvtColor(img_a, cv2.COLOR_BGR2RGB))
        axes[i, 0].set_title(f'📸 ANTES: {nome}\n{img_a.shape[1]}×{img_a.shape[0]}', fontsize=10)
        axes[i, 0].axis('off')

        axes[i, 1].imshow(cv2.cvtColor(img_d, cv2.COLOR_BGR2RGB))
        axes[i, 1].set_title(f'✅ DEPOIS: {nome}\n{img_d.shape[1]}×{img_d.shape[0]}', fontsize=10)
        axes[i, 1].axis('off')

    plt.tight_layout()
    plt.show()

---
## 6️⃣ Salvar CSV do Processamento

In [ ]:
csv_path = 'relatorio_redimensionamento.csv'
df_resultado.to_csv(csv_path, index=False)
print(f"✅ CSV salvo: {csv_path}")
print(f"   Linhas: {len(df_resultado)}")
print(f"   Colunas: {list(df_resultado.columns)}")

---
## 7️⃣ Relatório HTML Didático

Etapa **adicionada** ao script original. Gera um arquivo HTML standalone com:

- Cards de visão geral (total processado, redução de tamanho)
- Explicação visual do letterbox
- Tabela com todas as transformações
- Amostras lado a lado embutidas como base64
- Alertas se houve upscaling indevido

In [ ]:
def fig_para_b64(fig):
    """Converte figura matplotlib em data URI base64."""
    import io
    buf = io.BytesIO()
    fig.savefig(buf, format='png', dpi=95, bbox_inches='tight')
    buf.seek(0)
    return f"data:image/png;base64,{base64.b64encode(buf.read()).decode()}"


def img_para_b64(caminho, max_w=600):
    """Lê imagem do disco, redimensiona para preview e converte em base64."""
    img = cv2.imread(caminho)
    if img is None:
        return ""
    h, w = img.shape[:2]
    if w > max_w:
        nh = int(h * max_w / w)
        img = cv2.resize(img, (max_w, nh), interpolation=cv2.INTER_AREA)
    _, buf = cv2.imencode('.jpg', img, [cv2.IMWRITE_JPEG_QUALITY, 75])
    return f"data:image/jpeg;base64,{base64.b64encode(buf).decode()}"


def gerar_relatorio_html(saida, df, pasta_entrada, pasta_saida, tamanho_alvo):
    """Gera relatório HTML standalone do processamento."""
    df_ok = df[df['status'] == 'ok'].copy()
    n_total = len(df)
    n_ok = len(df_ok)
    n_erro = n_total - n_ok

    tam_antes = df_ok['tam_orig_kb'].sum() / 1024
    tam_depois = df_ok['tam_new_kb'].sum() / 1024
    reducao_pct = (1 - tam_depois / tam_antes) * 100 if tam_antes > 0 else 0
    n_upscale = int(df_ok['precisou_upscale'].sum())
    n_sem_barras = int((df_ok['tipo_barras'] == 'nenhuma').sum())

    # ===== Figura comparativa antes/depois (3 amostras) =====
    if len(df_ok) > 0:
        np.random.seed(42)
        n_am = min(3, len(df_ok))
        idx_am = np.random.choice(len(df_ok), n_am, replace=False)
        fig_a, axes = plt.subplots(n_am, 2, figsize=(13, 4*n_am))
        if n_am == 1:
            axes = axes.reshape(1, -1)
        for i, idx in enumerate(idx_am):
            nome = df_ok.iloc[idx]['arquivo']
            img_a = cv2.imread(os.path.join(pasta_entrada, nome))
            img_d = cv2.imread(os.path.join(pasta_saida, nome))
            axes[i,0].imshow(cv2.cvtColor(img_a, cv2.COLOR_BGR2RGB))
            axes[i,0].set_title(f'ANTES: {nome} ({img_a.shape[1]}x{img_a.shape[0]})')
            axes[i,0].axis('off')
            axes[i,1].imshow(cv2.cvtColor(img_d, cv2.COLOR_BGR2RGB))
            axes[i,1].set_title(f'DEPOIS: {nome} ({img_d.shape[1]}x{img_d.shape[0]})')
            axes[i,1].axis('off')
        plt.tight_layout()
        img_amostras = fig_para_b64(fig_a)
        plt.close(fig_a)
    else:
        img_amostras = ""

    # ===== Gráficos estatísticos =====
    fig_s, axes = plt.subplots(1, 3, figsize=(15, 4))
    x = np.arange(len(df_ok))
    axes[0].bar(x - 0.2, df_ok['tam_orig_kb'], 0.4, label='Antes', color='#c0392b')
    axes[0].bar(x + 0.2, df_ok['tam_new_kb'], 0.4, label='Depois', color='#27ae60')
    axes[0].set_xticks(x); axes[0].set_xticklabels([n[:10] for n in df_ok['arquivo']],
                                                    rotation=45, ha='right', fontsize=8)
    axes[0].set_ylabel('KB'); axes[0].set_title('Tamanho de arquivo')
    axes[0].legend(); axes[0].grid(axis='y', alpha=0.3)
    tipos = df_ok['tipo_barras'].value_counts()
    cores_pie = ['#3498db','#e67e22','#27ae60'][:len(tipos)]
    axes[1].pie(tipos.values, labels=tipos.index, autopct='%1.0f%%',
                colors=cores_pie, startangle=90)
    axes[1].set_title('Tipos de letterbox')
    prop_orig = df_ok['w_orig'] / df_ok['h_orig']
    prop_alvo = tamanho_alvo[0] / tamanho_alvo[1]
    axes[2].hist(prop_orig, bins=10, color='#9b59b6', alpha=0.7, edgecolor='black')
    axes[2].axvline(prop_alvo, color='red', linestyle='--', linewidth=2,
                    label=f'Alvo ({prop_alvo:.2f})')
    axes[2].set_xlabel('Proporção L/A'); axes[2].set_ylabel('Nº')
    axes[2].set_title('Proporções originais'); axes[2].legend()
    axes[2].grid(axis='y', alpha=0.3)
    plt.tight_layout()
    img_estat = fig_para_b64(fig_s)
    plt.close(fig_s)

    # ===== Linhas da tabela =====
    linhas = ""
    for _, r in df_ok.iterrows():
        prop_orig = r['w_orig'] / r['h_orig']
        upscale_badge = '<span class="b vermelho">UPSCALE</span>' if r['precisou_upscale'] else ''
        var_pct = (r['tam_new_kb'] - r['tam_orig_kb']) / r['tam_orig_kb'] * 100 if r['tam_orig_kb'] else 0
        var_str = f'{var_pct:+.0f}%'
        var_cor = '#27ae60' if var_pct < 0 else '#c0392b'
        linhas += f"""<tr>
        <td><b>{r['arquivo']}</b></td>
        <td>{r['w_orig']}x{r['h_orig']}</td>
        <td>{prop_orig:.2f}</td>
        <td>{r['w_new']}x{r['h_new']}</td>
        <td>{r['tipo_barras']} ({r['barras_px']}px)</td>
        <td style="text-align:right">{r['tam_orig_kb']:.0f} KB</td>
        <td style="text-align:right">{r['tam_new_kb']:.0f} KB</td>
        <td style="text-align:right;color:{var_cor};font-weight:600">{var_str}</td>
        <td>{upscale_badge}</td>
        </tr>"""

    # Aviso de upscaling
    aviso_upscale = ""
    if n_upscale > 0:
        aviso_upscale = f"""
        <div class="aviso laranja">
        <b>Atencao: {n_upscale} imagem(ns) precisaram de UPSCALING.</b>
        Quando a foto original e menor que o tamanho alvo ({tamanho_alvo[0]}x{tamanho_alvo[1]}),
        o script amplia ela usando cv2.INTER_AREA, que nao e ideal para ampliacao.
        Considere usar cv2.INTER_CUBIC ou cv2.INTER_LANCZOS4 para essas, OU revisar
        se vale a pena incluir fotos de baixa resolucao no dataset.
        </div>"""

    # Status semafórico geral
    if n_erro == 0 and n_upscale == 0:
        status_geral = ('VERDE', '#27ae60', 'Processamento limpo, sem erros nem upscaling')
    elif n_erro == 0:
        status_geral = ('AMARELO', '#f39c12', f'{n_upscale} imagem(ns) com upscaling — revisar')
    else:
        status_geral = ('VERMELHO', '#c0392b', f'{n_erro} erro(s) de leitura')

    html = """<!DOCTYPE html>
<html lang="pt-BR">
<head>
<meta charset="UTF-8">
<meta name="viewport" content="width=device-width,initial-scale=1.0">
<title>Relatorio Redimensionamento - Fotos Obra</title>
<style>
*{box-sizing:border-box;margin:0;padding:0}
body{font-family:-apple-system,BlinkMacSystemFont,"Segoe UI",Roboto,sans-serif;background:#f4f6fa;color:#2c3e50;line-height:1.6}
header{background:linear-gradient(135deg,#1f3864 0%,#2e5cb8 100%);color:white;padding:32px;box-shadow:0 2px 8px rgba(0,0,0,.1)}
header h1{font-size:24px;margin-bottom:6px}
header .sub{opacity:.85;font-size:14px}
.container{max-width:1200px;margin:0 auto;padding:32px}
.secao{margin-bottom:36px}
.secao h2{font-size:18px;margin-bottom:16px;color:#1f3864;border-left:4px solid #2e5cb8;padding-left:14px}
.secao p{margin-bottom:12px;color:#4a5568}
.grid-info{display:grid;grid-template-columns:repeat(4,1fr);gap:16px;margin-bottom:24px}
.card{background:white;border-radius:8px;padding:20px;box-shadow:0 1px 3px rgba(0,0,0,.08);text-align:center}
.card .t{font-size:11px;text-transform:uppercase;letter-spacing:.5px;color:#7f8c8d;margin-bottom:8px;font-weight:600}
.card .v{font-size:26px;font-weight:700;color:#2c3e50}
.card .s{font-size:11px;color:#95a5a6;margin-top:4px}
.figura{background:white;border-radius:8px;padding:20px;box-shadow:0 1px 3px rgba(0,0,0,.08);text-align:center}
.figura img{max-width:100%;height:auto}
table{width:100%;border-collapse:collapse;font-size:12px;background:white;border-radius:8px;overflow:hidden;box-shadow:0 1px 3px rgba(0,0,0,.08)}
th{background:#1f3864;color:white;padding:10px;text-align:left;font-weight:600;font-size:11px;text-transform:uppercase;letter-spacing:.3px}
td{padding:9px 10px;border-bottom:1px solid #ecf0f1}
tr:hover td{background:#f8f9fc}
.b{display:inline-block;padding:2px 7px;border-radius:4px;font-size:10px;font-weight:700;color:white}
.b.vermelho{background:#c0392b}
.aviso{padding:14px 18px;margin:14px 0;border-radius:6px;font-size:13px;border-left:4px solid}
.aviso.azul{background:#e8f4fd;border-left-color:#3498db}
.aviso.laranja{background:#fff8e1;border-left-color:#f39c12}
.aviso b{color:#1f3864}
.semaforo-box{background:white;border-radius:8px;padding:20px;text-align:center;box-shadow:0 1px 3px rgba(0,0,0,.08);border-top:4px solid __COR_STATUS__}
.semaforo-status{font-size:24px;font-weight:700;color:__COR_STATUS__;margin-bottom:6px}
.semaforo-desc{font-size:13px;color:#4a5568}
.explicacao{background:#f8f9fc;border-radius:8px;padding:20px;font-size:13px;line-height:1.8;color:#4a5568}
.explicacao code{background:#e8eef7;padding:2px 6px;border-radius:3px;font-family:"SF Mono",Consolas,monospace;font-size:12px;color:#1f3864}
footer{text-align:center;padding:24px;color:#95a5a6;font-size:12px;border-top:1px solid #ecf0f1;margin-top:32px}
@media(max-width:900px){.grid-info{grid-template-columns:repeat(2,1fr)}}
</style>
</head>
<body>
<header>
<h1>Relatorio de Redimensionamento - Fotos de Obra</h1>
<div class="sub">IPOS Pos-Graduacao | Profa M.Sc. Adriana Rolim | Emitido em __DATA__</div>
</header>
<div class="container">

<div class="secao">
<h2>Status Geral do Processamento</h2>
<div class="semaforo-box">
<div class="semaforo-status">__STATUS__</div>
<div class="semaforo-desc">__STATUS_DESC__</div>
</div>
</div>

<div class="secao">
<h2>Visao Geral</h2>
<div class="grid-info">
<div class="card"><div class="t">Total Processado</div><div class="v">__N_OK__</div><div class="s">de __N_TOTAL__ encontradas</div></div>
<div class="card"><div class="t">Tamanho Alvo</div><div class="v" style="font-size:20px">__TAM_ALVO__</div><div class="s">largura x altura</div></div>
<div class="card"><div class="t">Espaco em Disco</div><div class="v" style="color:__COR_REDUCAO__">__VAR_TAM__</div><div class="s">__TAM_ANTES__ MB &rarr; __TAM_DEPOIS__ MB</div></div>
<div class="card"><div class="t">Ja Estavam OK</div><div class="v">__N_SEM_BARRAS__</div><div class="s">sem barras de letterbox</div></div>
</div>
</div>

<div class="secao">
<h2>O que e Letterbox?</h2>
<div class="explicacao">
<p><b>Letterbox</b> e a tecnica de redimensionar uma imagem para um tamanho alvo
<b>mantendo a proporcao original</b>, preenchendo as areas sobrando com
faixas pretas (como em filmes antigos exibidos em TVs widescreen).</p>
<p>Para visao computacional em construcao civil, isso e fundamental: se
distorcessemos as fotos (resize direto), uma viga circular viraria eliptica,
e o modelo aprenderia geometrias erradas. Com letterbox, todas as fotos ficam
no mesmo tamanho final (<code>__TAM_ALVO__</code>) mas <b>preservam a forma real</b>
dos objetos.</p>
<p><b>Como funciona o calculo:</b></p>
<ul style="margin-left:24px;line-height:2">
<li>Calcula <code>fator_largura = largura_alvo / largura_original</code></li>
<li>Calcula <code>fator_altura = altura_alvo / altura_original</code></li>
<li>Pega o <b>menor fator</b> (garante que cabe inteiro)</li>
<li>Redimensiona com esse fator</li>
<li>Centraliza e preenche bordas pretas (<code>cv2.copyMakeBorder</code>)</li>
</ul>
</div>
</div>

__AVISO_UPSCALE__

<div class="secao">
<h2>Estatisticas Visuais</h2>
<div class="figura">__IMG_ESTAT__</div>
</div>

<div class="secao">
<h2>Amostras: Antes &times; Depois</h2>
<div class="figura">__IMG_AMOSTRAS__</div>
</div>

<div class="secao">
<h2>Detalhamento por Imagem</h2>
<table>
<thead><tr>
<th>Arquivo</th><th>Original</th><th>Prop.</th><th>Final</th>
<th>Letterbox</th><th style="text-align:right">Tam. antes</th><th style="text-align:right">Tam. depois</th>
<th style="text-align:right">&Delta;</th><th>Obs</th>
</tr></thead>
<tbody>__LINHAS__</tbody>
</table>
</div>

<div class="secao">
<h2>Proximos Passos</h2>
<div class="aviso azul">
<b>Apos este redimensionamento, as fotos estao prontas para:</b>
<ol style="margin-left:24px;line-height:2">
<li>Treinar/aplicar modelos de visao computacional (U-Net, YOLO, ResNet)</li>
<li>Comparar com renders BIM da mesma fachada (alinhamento por homografia/PnP)</li>
<li>Compor mosaicos e timelines de evolucao da obra</li>
<li>Anexar a parecer tecnico com padronizacao visual</li>
</ol>
</div>
</div>

</div>
<footer>
Gerado automaticamente | Script de Redimensionamento de Fotos de Obra<br>
IPOS Pos-Graduacao - Profa M.Sc. Adriana Rolim
</footer>
</body>
</html>"""

    subs = {
        '__DATA__': datetime.now().strftime('%d/%m/%Y %H:%M'),
        '__STATUS__': status_geral[0],
        '__COR_STATUS__': status_geral[1],
        '__STATUS_DESC__': status_geral[2],
        '__N_OK__': str(n_ok),
        '__N_TOTAL__': str(n_total),
        '__TAM_ALVO__': f'{tamanho_alvo[0]} x {tamanho_alvo[1]}',
        '__VAR_TAM__': f'{abs(reducao_pct):.0f}% {"&darr;" if reducao_pct>0 else "&uarr;"}',
        '__COR_REDUCAO__': '#27ae60' if reducao_pct > 0 else '#c0392b',
        '__TAM_ANTES__': f'{tam_antes:.1f}',
        '__TAM_DEPOIS__': f'{tam_depois:.1f}',
        '__N_SEM_BARRAS__': str(n_sem_barras),
        '__AVISO_UPSCALE__': aviso_upscale,
        '__IMG_ESTAT__': f'<img src="{img_estat}" alt="Estatisticas">',
        '__IMG_AMOSTRAS__': f'<img src="{img_amostras}" alt="Amostras antes/depois">',
        '__LINHAS__': linhas,
    }
    for k, v in subs.items():
        html = html.replace(k, str(v))

    Path(saida).write_text(html, encoding='utf-8')
    return saida


html_path = 'relatorio_redimensionamento.html'
gerar_relatorio_html(html_path, df_resultado, PASTA_ENTRADA, PASTA_SAIDA, TAMANHO_ALVO)
print(f"✅ Relatório HTML gerado: {html_path}")
print(f"   Tamanho: {os.path.getsize(html_path):,} bytes".replace(',', '.'))

# Download no Colab
try:
    from google.colab import files
    print("\n💡 Para baixar, descomente as linhas abaixo:")
    print(f"# files.download('{html_path}')")
    print(f"# files.download('relatorio_redimensionamento.csv')")
except ImportError:
    pass

---
## ✅ Resumo dos Outputs

| Arquivo | Conteúdo |
|---|---|
| `fotos_obra_redimensionadas/` (no Drive) | Todas as fotos padronizadas em 1920×1080 |
| `relatorio_redimensionamento.csv` | Tabela com cada transformação (dimensões antes/depois, tamanho de arquivo, letterbox aplicado) |
| `relatorio_redimensionamento.html` | Relatório visual standalone com cards, gráficos, amostras antes×depois e tabela detalhada |

### Próximos passos sugeridos

1. **Verificar visualmente** algumas amostras na pasta de saída do Drive
2. **Conferir o relatório HTML** para identificar imagens que precisaram de upscaling
3. **Aplicar o próximo pré-processamento** da sequência (redução de ruído, contraste, etc) usando esse mesmo dataset padronizado
4. **Usar essas fotos** como entrada para o pipeline BIM 4D + CV (notebook anterior)

### Modificações comuns que você pode querer

- **Trocar o tamanho alvo:** ajuste `TAMANHO_ALVO` no início (ex: `(512, 512)` para U-Net)
- **Cor da borda:** mudar `value=(0, 0, 0)` para `(255, 255, 255)` (branco) ou cor da paleta da obra
- **Não fazer upscaling:** adicionar verificação `if w >= largura_alvo or h >= altura_alvo` antes do resize
- **Salvar com JPEG quality customizada:** `cv2.imwrite(..., [cv2.IMWRITE_JPEG_QUALITY, 90])`

---

> **Profª M.Sc. Adriana Rolim** — IPOS Pós-Graduação
> *Redimensionamento de Fotos de Obra com Letterbox*